# 05 Dashboard Evaluation and Business Recommendations

This notebook documents the final interactive dashboard developed for the Airbnb revenue analytics project. The dashboard brings together the exploratory analysis, predictive modelling and business interpretation completed in the previous notebooks.

The application was developed in Streamlit and connects directly to the cleaned Airbnb dataset, the final tuned Random Forest model and the aggregated feature-importance results. Its purpose is to provide an accessible decision-support interface for exploring market performance and estimating annual Airbnb revenue.

## 1. Dashboard Purpose

The dashboard was designed to translate the technical outputs of the project into a format that can be used by non-technical decision-makers.

It supports three main tasks:

1. exploring Airbnb market performance using interactive filters and visualisations;
2. estimating annual revenue for a potential listing using the final predictive model;
3. interpreting the main factors that influence predicted revenue.

The dashboard is intended for prospective hosts, property investors and short-term rental managers who want to compare market conditions and evaluate potential listing performance.

## 2. Dashboard Structure

The dashboard is organised into five main sections.

### Executive Market Overview

The first section provides high-level key performance indicators for the current filter selection. These include the number of listings, average annual revenue, average daily rate (ADR), Superhost proportion and median annual revenue.

Including both average and median revenue is important because annual revenue is strongly right-skewed. The median therefore provides a more robust indication of a typical listing than the arithmetic mean alone.

### Market Explorer

The Market Explorer contains four interactive visualisations:

- annual revenue distribution;
- average revenue by property type;
- average revenue by city;
- ADR distribution.

The charts update automatically when the user changes the city, property-type or listing-type filters.

Extreme revenue and ADR observations above the 99th percentile are excluded from the respective histogram displays only. The underlying dataset and calculations remain unchanged. This prevents a small number of unusually large values from compressing the main distributions.

Property types represented by fewer than 30 listings are excluded from the property-type comparison to reduce the risk of presenting unstable averages based on very small samples.

## 3. Revenue Estimator

The Revenue Estimator provides an interactive interface to the final tuned Random Forest model.

Users can modify listing characteristics including:

- city;
- property type;
- listing type;
- cancellation policy;
- bedrooms and bathrooms;
- maximum guest capacity;
- average daily rate;
- listing age;
- host portfolio size;
- Superhost status;
- selected amenities.

The model predicts log-transformed annual revenue internally. The prediction is then transformed back to US dollars using the inverse logarithmic transformation.

The estimator initially displays the characteristics of a real listing from the dataset rather than an arbitrary hypothetical property. This reduces the likelihood of beginning with an unrealistic combination of features and reflects the fact that tree-based models are most reliable for observations similar to those represented in the training data.

In [10]:
import numpy as np
import pandas as pd
from pathlib import Path

In [11]:
# The dashboard converts the user-entered ADR to the same
# transformed feature used during model training.

example = pd.DataFrame({
    "Average Daily Rate (USD)": [75, 150, 250]
})

example["Log ADR Used by Model"] = np.log1p(
    example["Average Daily Rate (USD)"]
)

example

,Average Daily Rate (USD),Log ADR Used by Model
0,75,4.330733
1,150,5.017280
2,250,5.525453


### Comparable-Market Benchmark

A model prediction alone is difficult to interpret without a relevant benchmark.

The dashboard therefore compares the estimated revenue with the median annual revenue of comparable listings. The first comparison group matches:

- city;
- property type;
- listing type.

If fewer than 30 comparable listings are available, the benchmark is broadened first to listings with the same city and listing type and, if necessary, to all listings within the selected city.

The median is used instead of the mean because Airbnb revenue is heavily right-skewed and contains extreme high-revenue listings.

This benchmark should not be interpreted as a formal prediction interval. It is included to give the user practical market context for the model estimate.

## 4. Model Performance Presented in the Dashboard

The dashboard includes model-performance information to prevent the prediction tool from appearing more precise than the underlying model actually is.

The final tuned Random Forest produced approximately:

- Test R² on the log-transformed target: **0.504**
- Cross-validated mean R²: **0.502**
- Test MAE in original US-dollar terms: **$9,828**

The three-fold cross-validation results were highly consistent, with R² values of approximately 0.500 to 0.504. This indicates that the model's performance is stable across different subsets of the data.

However, an R² of approximately 0.50 also means that a substantial proportion of revenue variation remains unexplained. Predictions should therefore be interpreted as decision-support estimates rather than guaranteed financial outcomes.

In [12]:
cross_validation_display = pd.DataFrame({
    "Fold": [1, 2, 3],
    "R² (Log Scale)": [0.5008, 0.4997, 0.5040],
    "MAE (Log Scale)": [0.8459, 0.8418, 0.8411],
    "RMSE (Log Scale)": [1.1157, 1.1086, 1.1093]
})

cross_validation_display.round(4)

,Fold,R² (Log Scale),MAE (Log Scale),RMSE (Log Scale)
0,1,0.5008,0.8459,1.1157
1,2,0.4997,0.8418,1.1086
2,3,0.5040,0.8411,1.1093


## 5. Key Revenue Drivers

The dashboard also displays the aggregated feature importance from the final tuned Random Forest model.

The strongest predictors were:

1. Average Daily Rate;
2. Listing Age;
3. Host Listing Count;
4. City;
5. Cancellation Policy;
6. Property Type;
7. Maximum Guests;
8. Superhost status;
9. Bedrooms;
10. selected amenities and other listing characteristics.

Average Daily Rate was by far the strongest individual predictor, followed by Listing Age.

Feature importance should be interpreted carefully. It indicates how strongly the Random Forest relies on a variable when making predictions, but it does not demonstrate that changing that variable will directly cause revenue to increase or decrease.

In [13]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent

feature_importance = pd.read_csv(
    PROJECT_ROOT / "tables" / "table_23_random_forest_feature_importance.csv"
)

feature_importance.head(10)

,Feature,Importance
0,Log ADR,0.407191
1,Listing Age (Days),0.220868
2,Host Listing Count,0.075090
3,Max Guests,0.031694
4,Cancellation Policy,0.031633
5,City,0.021770
6,Airbnb Superhost,0.015831
7,Bedrooms,0.014007
8,Airbnb Superhost,0.013730
9,Has Parking,0.012756


## 6. Business Insights and Recommendations

### Pricing Strategy

Average Daily Rate is the strongest predictor in the final model. Pricing decisions should therefore be benchmarked against comparable properties within the same city and listing segment rather than using a single pricing approach across all markets.

At the same time, the model results should not be interpreted as evidence that continually increasing ADR will automatically increase revenue. Random Forest predictions are not constrained to follow a monotonic relationship, and revenue also depends on occupancy and other listing characteristics.

### Listing Maturity

Listing Age is the second most important predictor. Established listings may benefit from accumulated reviews, visibility, platform history and improved host operating experience.

For a new host or investor, this suggests that first-year performance should not automatically be expected to match that of mature listings.

### Host Portfolio

Host Listing Count also contributes materially to the model. This may reflect differences between individual hosts and professional operators managing multiple properties.

However, feature importance alone does not establish that increasing the number of properties managed will cause an individual listing's revenue to rise.

### Local Market Conditions

City, property type and cancellation policy all influence predicted revenue. This supports the conclusion that investment decisions should be market-specific.

A listing configuration that performs strongly in one city may not generate the same outcome in another market.

### Model-Based Decision Support

The model should be used as one input to an investment decision rather than as a stand-alone financial forecast.

Other factors such as regulation, seasonality, local events, competition, property costs and changing demand are not fully represented in the model.

## 7. Dashboard and Model Limitations

Several limitations should be considered when interpreting the dashboard.

### Geographic Coverage

The dataset covers nine international cities. Results therefore cannot automatically be generalised to markets that are not represented in the training data.

### Historical Data

The model learns relationships from historical observations. Airbnb demand, regulations and pricing conditions can change over time, meaning future performance may differ from historical patterns.

### Prediction Accuracy

The final model explains approximately half of the variation in log-transformed annual revenue. Residual analysis also showed that prediction errors become substantially larger for exceptional high-revenue listings.

The model tends to be more reliable for listings similar to those represented frequently in the training data.

### Revenue Skewness

Annual revenue is strongly right-skewed. Although logarithmic transformation improved modelling performance, converting predictions back to the original dollar scale can still lead to conservative predictions for some high-revenue properties.

### Missing External Variables

The model does not directly include all factors that may influence Airbnb performance, such as:

- local regulations;
- tourism events;
- seasonality;
- distance to major attractions;
- neighbourhood characteristics;
- competitor pricing;
- property acquisition and operating costs.

### Feature Importance

Random Forest feature importance measures predictive contribution rather than causality. Business recommendations should therefore be interpreted alongside the exploratory analysis rather than as direct causal claims.

## 8. Potential Future Improvements

If the dashboard were developed beyond the scope of this academic project, several improvements could be considered.

Future development could include:

- integration with regularly updated or live market data;
- neighbourhood-level geographic analysis;
- monthly or seasonal revenue forecasting;
- occupancy modelling alongside ADR;
- prediction intervals or uncertainty estimates;
- scenario comparison between multiple potential properties;
- interactive mapping;
- integration of regulatory and tourism indicators;
- additional model families such as gradient-boosted trees;
- automatic model retraining when new data becomes available.

For a production investment application, model behaviour should also be tested more extensively on hypothetical listing combinations and monitored for unrealistic predictions outside well-represented areas of the training data.

In [14]:
# ---------------------------------------------------------
# Final project artefact check
# ---------------------------------------------------------

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

artefacts = {
    "Cleaned dataset": (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "airbnb_cleaned.csv"
    ),
    "Final model": (
        PROJECT_ROOT
        / "models"
        / "airbnb_revenue_model.joblib"
    ),
    "Feature importance": (
        PROJECT_ROOT
        / "tables"
        / "table_23_random_forest_feature_importance.csv"
    ),
    "Dashboard application": (
        PROJECT_ROOT
        / "dashboard"
        / "app.py"
    )
}

artefact_check = pd.DataFrame({
    "Artefact": artefacts.keys(),
    "Available": [
        path.exists()
        for path in artefacts.values()
    ],
    "Path": [
        str(path)
        for path in artefacts.values()
    ]
})

artefact_check

,Artefact,Available,Path
0,Cleaned dataset,True,c:\Users\Probook\NCIRL 2025-2026\DA for Busine...
1,Final model,True,c:\Users\Probook\NCIRL 2025-2026\DA for Busine...
2,Feature importance,True,c:\Users\Probook\NCIRL 2025-2026\DA for Busine...
3,Dashboard application,True,c:\Users\Probook\NCIRL 2025-2026\DA for Busine...


In [15]:
dashboard_features = pd.DataFrame({

    "Dashboard Component":[
        "Executive Market Overview",
        "Market Explorer",
        "Revenue Estimator",
        "Model Performance",
        "Key Revenue Drivers",
        "Business Insights"
    ],

    "Purpose":[
        "High-level KPIs",
        "Interactive visualisations",
        "Revenue prediction",
        "Model transparency",
        "Model interpretation",
        "Business recommendations"
    ]

})

dashboard_features

,Dashboard Component,Purpose
0,Executive Market Overview,High-level KPIs
1,Market Explorer,Interactive visualisations
2,Revenue Estimator,Revenue prediction
3,Model Performance,Model transparency
4,Key Revenue Drivers,Model interpretation
5,Business Insights,Business recommendations


In [16]:
dashboard_users = pd.DataFrame({

    "User":[
        "Property Investors",
        "Airbnb Hosts",
        "Portfolio Managers",
        "Market Analysts"
    ],

    "Primary Benefit":[
        "Evaluate investment opportunities",
        "Estimate listing revenue",
        "Compare property performance",
        "Explore market trends"
    ]

})

dashboard_users

,User,Primary Benefit
0,Property Investors,Evaluate investment opportunities
1,Airbnb Hosts,Estimate listing revenue
2,Portfolio Managers,Compare property performance
3,Market Analysts,Explore market trends


In [17]:
future_features = pd.DataFrame({

    "Potential Enhancement":[

        "Live Airbnb data",

        "Interactive maps",

        "Monthly forecasting",

        "Scenario comparison",

        "Prediction intervals",

        "Automatic model retraining"

    ],

    "Business Value":[

        "Real-time market insights",

        "Neighbourhood analysis",

        "Seasonal planning",

        "Investment evaluation",

        "Decision confidence",

        "Improved long-term accuracy"

    ]

})

future_features

,Potential Enhancement,Business Value
0,Live Airbnb data,Real-time market insights
1,Interactive maps,Neighbourhood analysis
2,Monthly forecasting,Seasonal planning
3,Scenario comparison,Investment evaluation
4,Prediction intervals,Decision confidence
5,Automatic model retraining,Improved long-term accuracy


## Final Remarks

This notebook evaluated the completed dashboard developed for the Airbnb revenue analytics project.

The dashboard integrates exploratory data analysis, machine learning and business interpretation into a single interactive application that supports practical decision-making.

Although the final Random Forest model explains approximately half of the variation in annual revenue, it demonstrates stable performance across cross-validation folds and provides useful decision-support estimates when interpreted alongside comparable market benchmarks and domain knowledge.

The completed application represents the final deliverable of the technical component of the project.